# NB2 v7 — Análise: Detecção e Classificação de Anomalias Portuárias

**Pipeline:** Modelagem XGBoost + correção AR(1) → Diagnóstico de resíduos → Detecção ensemble → Classificação dual score → Fingerprint ANTAQ

Carrega parquets do NB1, processa em memória. Resultados exportados para NB3 (impacto econômico).

### Seções
| Seção | Conteúdo |
|---|---|
| **A** | Modelagem preditiva (XGBoost + correção AR(1) nos resíduos) |
| **B** | Diagnóstico de resíduos e justificativa AR(1) |
| **C** | Detecção de anomalias (ensemble MAD + STL + IForest sobre inovações) |
| **D** | Classificação (Score A co-ocorrência + Score B índice global) |
| **E** | Fingerprint ANTAQ e classificador endógeno |
| **F** | Visualizações e estudos de caso |
| **G** | Exportação de resultados |

**v7:** Features restritas à própria dimensão-alvo por modelo (`get_feat_cols_for(dim)`). Passe 2 (XGBoost com lags de resíduo) substituído por filtro AR(1): inovações $\varepsilon_t = r_t - \hat{\rho} \cdot r_{t-1}$ usadas na detecção.

In [ ]:
# Célula 0 — Setup e carregar dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import sys; sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import *

plt.rcParams.update({"figure.figsize": (14, 5), "figure.dpi": 100})

# Diretório de saída
OUT = ROOT / "data" / "output"
OUT.mkdir(parents=True, exist_ok=True)

# Carregar outputs do NB1
feat = pd.read_parquet(DATA_PROC / "features_semanal.parquet")
weekly = pd.read_parquet(DATA_PROC / "series_semanal.parquet")
gi = pd.read_parquet(DATA_PROC / "indice_global.parquet")

PORTOS = sorted(feat["porto"].unique())
print(f"Portos: {len(PORTOS)}, Features shape: {feat.shape}")
print(f"Semanal shape: {weekly.shape}, Índice global: {len(gi)} semanas")

## ═══ SEÇÃO A: MODELAGEM PREDITIVA ═══

Objetivo: gerar resíduos $r_t = y_t - \hat{y}_t$ e inovações $\varepsilon_t = r_t - \hat{\rho} \cdot r_{t-1}$ para cada porto × dimensão.

**Abordagem em duas etapas:**
1. **XGBoost** com features temporais own-only (~15 por dimensão) → captura tendência, sazonalidade, calendário → resíduos P1
2. **AR(1)** nos resíduos P1 → estima $\hat{\rho}$ por porto × dim → inovações $\varepsilon_t \approx$ ruído branco

A detecção de anomalias (Seção C) opera sobre as **inovações**, removendo a dependência serial de curto prazo.

**Seleção de modelo:** melhor modelo por *dimensão* (não por porto) via CV expanding window. Mesma arquitetura aplicada a todos os portos, coeficientes individuais.


In [ ]:
# Célula A1 — Feature columns e modelos candidatos
# v6: Features restritas à própria dimensão-alvo + calendário (sem features cruzadas)

CALENDAR_COLS = [c for c in feat.columns if c in ("month_sin", "month_cos", "week_of_year")]

def get_feat_cols_for(df, target_dim):
    """Features temporais da própria dimensão-alvo + calendário.
    Exclui features de outras dimensões intencionalmente: o objetivo do modelo
    é capturar a dinâmica 'normal' de cada variável para que perturbações
    inter-variáveis apareçam como anomalias nos resíduos, em vez de serem
    absorvidas pela predição."""
    own = [c for c in df.columns
           if c.startswith(target_dim + "_") and "stl" not in c and "resid" not in c]
    return own + CALENDAR_COLS

# Verificar
for dim in DIMS:
    cols = get_feat_cols_for(feat, dim)
    print(f"  {dim}: {len(cols)} features → {cols}")

MODEL_GRID = {
    "naive52":      {"type": "naive", "params": {"lag": 52}},
    "xgb_shallow":  {"type": "xgboost", "params": {"max_depth": 4, "n_estimators": 200, "learning_rate": 0.05}},
    "xgb_deep":     {"type": "xgboost", "params": {"max_depth": 6, "n_estimators": 300, "learning_rate": 0.03}},
    "lgbm_shallow": {"type": "lightgbm", "params": {"max_depth": 4, "n_estimators": 200, "learning_rate": 0.05}},
}


In [ ]:
# Célula A2 — Funções de treino/predição

def train_predict(mname, mconf, y, X, tr, pr, dates=None):
    """Treina nos tr, prediz nos pr. Retorna array de predições."""
    preds = np.full(len(pr), np.nan)
    y_tr = y[tr]
    mtype = mconf.get("type", mname)
    params = mconf.get("params", {})

    if np.isnan(y_tr).mean() > 0.3:
        return preds

    if mtype == "naive":
        lag = params.get("lag", 52)
        for i, idx in enumerate(pr):
            if idx >= lag:
                preds[i] = y[idx - lag]

    elif mtype in ("xgboost", "lightgbm"):
        X_tr, X_pr = X[tr], X[pr]
        ok_tr = ~(np.any(np.isnan(X_tr), 1) | np.isnan(y_tr))
        ok_pr = ~np.any(np.isnan(X_pr), 1)
        if ok_tr.sum() < 20 or ok_pr.sum() == 0:
            return preds
        Cls = XGBRegressor if mtype == "xgboost" else LGBMRegressor
        extra = {"verbosity": 0} if mtype == "xgboost" else {"verbose": -1}
        m = Cls(**params, random_state=42, **extra)
        m.fit(X_tr[ok_tr], y_tr[ok_tr])
        preds[ok_pr] = m.predict(X_pr[ok_pr])

    return preds


def cv_expanding(y, X, dates, mname, mconf):
    """Cross-validation expanding window."""
    n = len(y); maes = []; te = CV_MIN_TRAIN_WEEKS
    while te + CV_VAL_WEEKS <= n:
        tr = np.arange(0, te); va = np.arange(te, te + CV_VAL_WEEKS)
        p = train_predict(mname, mconf, y, X, tr, va, dates)
        valid = ~(np.isnan(p) | np.isnan(y[va]))
        if valid.sum() >= 5:
            maes.append(np.mean(np.abs(y[va][valid] - p[valid])))
        te += CV_STEP_WEEKS
    return maes


def walkforward(y, X, dates, mname, mconf):
    """Walk-forward com retraining periódico + naive fallback para NaN."""
    n = len(y)
    preds = np.full(n, np.nan)
    te = CV_MIN_TRAIN_WEEKS

    while te < n:
        pe = min(te + WALKFORWARD_RETRAIN_WEEKS, n)
        tr = np.arange(0, te)
        pr = np.arange(te, pe)
        preds[pr] = train_predict(mname, mconf, y, X, tr, pr, dates)
        te = pe

    # ── NAIVE FALLBACK: preencher NaN remanescentes com lag-52 ──
    # Cobre: (a) período inicial antes do walk-forward começar
    #         (b) semanas onde XGBoost retorna NaN por features faltantes
    n_nan_before = np.isnan(preds).sum()
    for i in range(len(preds)):
        if np.isnan(preds[i]) and i >= 52 and not np.isnan(y[i - 52]):
            preds[i] = y[i - 52]
    n_nan_after = np.isnan(preds).sum()

    if n_nan_before > n_nan_after:
        print(f"    Fallback naive: {n_nan_before - n_nan_after} NaN preenchidos "
              f"({n_nan_after} restantes)")

    return preds

print("Funções definidas: train_predict, cv_expanding, walkforward (com naive fallback)")

In [ ]:
# Célula A3 — Cross-validation (Passe 1)
# v6: X construído dentro do loop de dims (features específicas por dimensão)
# PATCH 1: manter TODAS as 4 dimensões. Se nenhum ML bater naive, usar naive.

cv_all = []
for porto in PORTOS:
    df_p = feat[feat["porto"] == porto].sort_values("date").reset_index(drop=True)
    dates = df_p["date"].values
    for dim in DIMS:
        cols = get_feat_cols_for(df_p, dim)   # v6: features da própria dim
        X = df_p[cols].values
        y = df_p[dim].values
        for mname, mconf in MODEL_GRID.items():
            maes = cv_expanding(y, X, dates, mname, mconf)
            if maes:
                cv_all.append({"porto": porto, "dim": dim, "modelo": mname,
                               "mae_mean": np.mean(maes), "n_folds": len(maes)})
    print(f"  CV {porto}: OK")

cv_df = pd.DataFrame(cv_all)

# ── PATCH 1: Melhor modelo por dimensão — FORÇAR todas as 4 dims ──
best_by_dim = []
for dim in DIMS:
    dim_cv = cv_df[cv_df["dim"] == dim]
    if dim_cv.empty:
        print(f"  ⚠️ {dim}: sem resultados de CV, usando naive52")
        best_by_dim.append({"dim": dim, "modelo": "naive52", "mae_mean": np.nan})
        continue

    best = dim_cv.groupby("modelo")["mae_mean"].mean().reset_index()
    best = best.sort_values("mae_mean")
    winner = best.iloc[0]

    naive_mae = best[best["modelo"] == "naive52"]["mae_mean"].values
    naive_mae = naive_mae[0] if len(naive_mae) > 0 else np.inf

    improvement = 1 - winner["mae_mean"] / naive_mae if naive_mae > 0 else 0
    print(f"  {dim:20s}: {winner['modelo']:15s} MAE={winner['mae_mean']:.2f} "
          f"(↓{improvement:.0%} vs naive)")

    best_by_dim.append({
        "dim": dim,
        "modelo": winner["modelo"],
        "mae_mean": winner["mae_mean"]
    })

best_by_dim = pd.DataFrame(best_by_dim)
print(f"\nDimensões modeladas: {best_by_dim['dim'].tolist()}")
assert len(best_by_dim) == len(DIMS), f"ERRO: {len(best_by_dim)} dims, esperado {len(DIMS)}"


### A4 — Passe 1: Walk-forward

Predição walk-forward com retraining periódico. Semanas sem dados de treino suficientes usam **fallback naive** (lag-52), que cobre:
- **Período inicial:** primeiras ~`CV_MIN_TRAIN_WEEKS` semanas antes de acumular treino suficiente
- **Features faltantes:** gaps em variáveis exógenas (ex: ANTAQ com cobertura parcial)

Os "553 NaN preenchidos" que aparecem no output **não significam** que 91% do modelo é naive — significam que 553 semanas do início da série (onde o walk-forward ainda não iniciou) foram cobertas pelo naive. As 52 restantes = primeiro ano sem lag-52 disponível.


In [ ]:
# Célula A4 — Passe 1: walk-forward resíduos
# v6: X construído dentro do loop de dims (features específicas por dimensão)
# Fix 1: usa DIMS (todas as 4), não DIMS_OK

all_p1 = []
for porto in PORTOS:
    df_p = feat[feat["porto"] == porto].sort_values("date").reset_index(drop=True)
    dates = df_p["date"].values
    for dim in DIMS:
        cols = get_feat_cols_for(df_p, dim)   # v6: features da própria dim
        X = df_p[cols].values
        row = best_by_dim[best_by_dim["dim"] == dim].iloc[0]
        y = df_p[dim].values
        preds = walkforward(y, X, dates, row["modelo"], MODEL_GRID[row["modelo"]])
        all_p1.append(pd.DataFrame({
            "date": dates, "porto": porto, "dim": dim,
            "y_true": y, "y_pred": preds, "residual": y - preds}))
    print(f"  P1 {porto}: OK")

resid_p1 = pd.concat(all_p1, ignore_index=True)
print(f"\nResíduos P1: {len(resid_p1):,} ({len(DIMS)} dims × {len(PORTOS)} portos)")


### A4b — Feature Importance do XGBoost por dimensão

Retreina o melhor modelo (fit final) para cada porto × dimensão sobre os dados completos e extrai `feature_importances_` (gain). Importâncias médias entre portos mostram quais features são mais relevantes para prever cada variável operacional.

In [ ]:
# Célula A4b — Feature importance do XGBoost por dimensão
# Retreina best model (final fit) por porto × dim, acumula feature_importances_.

fi_rows = []
for dim in DIMS:
    row = best_by_dim[best_by_dim["dim"] == dim].iloc[0]
    mname = row["modelo"]
    mconf = MODEL_GRID[mname]
    mtype = mconf.get("type", mname)

    if mtype not in ("xgboost", "lightgbm"):
        print(f"  {dim}: modelo {mname} não é tree-based → sem importâncias")
        continue

    for porto in PORTOS:
        df_p = feat[feat["porto"] == porto].sort_values("date").reset_index(drop=True)
        cols = get_feat_cols_for(df_p, dim)
        X = df_p[cols].values
        y = df_p[dim].values
        ok = ~(np.any(np.isnan(X), 1) | np.isnan(y))
        if ok.sum() < 50:
            continue

        Cls = XGBRegressor if mtype == "xgboost" else LGBMRegressor
        extra = {"verbosity": 0} if mtype == "xgboost" else {"verbose": -1}
        m = Cls(**mconf.get("params", {}), random_state=42, **extra)
        m.fit(X[ok], y[ok])

        for col, imp in zip(cols, m.feature_importances_):
            fi_rows.append({"dim": dim, "porto": porto, "feature": col, "importance": imp})

fi_df = pd.DataFrame(fi_rows)
print(f"Feature importances coletadas: {len(fi_df)} linhas "
      f"({fi_df['dim'].nunique()} dims × {fi_df['porto'].nunique()} portos)")

# ── Plot: média das importâncias por dimensão ──
n_dims = len(DIMS)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, dim in zip(axes.flatten(), DIMS):
    sub = fi_df[fi_df["dim"] == dim]
    if sub.empty:
        ax.set_title(f"{DIM_LABELS.get(dim, dim)} (naive — sem importâncias)")
        continue
    mean_imp = sub.groupby("feature")["importance"].mean().sort_values(ascending=True)
    mean_imp.plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"Feature Importance — {DIM_LABELS.get(dim, dim)}", fontsize=11)
    ax.set_xlabel("Importância média (gain)")
    ax.set_ylabel("")

plt.suptitle("XGBoost Feature Importance por Dimensão\n(média entre portos, fit final)",
             fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "feature_importance_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Top-3 features por dimensão ──
print(f"\nTop-3 features por dimensão:")
for dim in DIMS:
    sub = fi_df[fi_df["dim"] == dim].groupby("feature")["importance"].mean()
    top3 = sub.sort_values(ascending=False).head(3)
    if len(top3) > 0:
        print(f"  {dim:22s}: {', '.join(f'{f} ({v:.3f})' for f, v in top3.items())}")

### A5 — Correção AR(1) dos resíduos

Os resíduos do Passe 1 apresentam autocorrelação substancial (ACF(1) mediana ~0.37). Em vez de um segundo passe de XGBoost (que não melhora a ACF — testado em v5 e v6), aplicamos um filtro AR(1): $\varepsilon_t = r_t - \hat{\rho} \cdot r_{t-1}$.

Cada porto × dimensão recebe seu próprio $\hat{\rho}$, estimado pela autocorrelação amostral. As inovações $\varepsilon_t$ são aproximadamente ruído branco (ACF(1) ≈ 0), permitindo que o ensemble de detecção opere sobre desvios genuinamente independentes.

**v8:** Após a correção AR(1), as primeiras 78 semanas (~1.5 anos) de cada série são removidas (burn-in) para estabilização do modelo XGBoost antes da detecção de anomalias.


In [ ]:
# Célula A5 — Correção AR(1) dos resíduos do Passe 1
# O XGBoost captura tendência e sazonalidade; o AR(1) captura a
# dependência serial remanescente nos resíduos.
# Detecção (Seção C) opera sobre as inovações ε_t.

from statsmodels.tsa.stattools import acf

ar1_coefs = {}
all_corrected = []

for porto in PORTOS:
    for dim in DIMS:
        mask = (resid_p1["porto"] == porto) & (resid_p1["dim"] == dim)
        df_rd = resid_p1.loc[mask].sort_values("date").copy()
        r = df_rd["residual"].values

        # Estimar ρ̂ via autocorrelação amostral
        r_clean = r[~np.isnan(r)]
        if len(r_clean) > 52:
            rho = acf(r_clean, nlags=1, fft=True)[1]
        else:
            rho = 0.0

        ar1_coefs[(porto, dim)] = rho

        # Inovação: ε_t = r_t − ρ̂·r_{t-1}
        r_shifted = np.roll(r, 1)
        r_shifted[0] = np.nan
        innovation = r - rho * r_shifted

        df_rd["innovation"] = innovation
        df_rd["rho_ar1"] = rho
        all_corrected.append(df_rd)

    print(f"  AR(1) {porto}: OK")

resid = pd.concat(all_corrected, ignore_index=True)

# Diagnóstico AR(1)
rho_vals = list(ar1_coefs.values())
print(f"\nResíduos com AR(1): {len(resid):,}")
print(f"\nCoeficientes AR(1):")
print(f"  ρ̂ mediano: {np.median(rho_vals):.3f}")
print(f"  ρ̂ médio:   {np.mean(rho_vals):.3f}")
print(f"  ρ̂ > 0.30:  {sum(1 for v in rho_vals if v > 0.30)}/{len(rho_vals)}")

# Verificar ACF(1) das inovações (ANTES de sobrescrever "residual")
acf_before, acf_after = [], []
for porto in PORTOS:
    for dim in DIMS:
        mask = (resid["porto"] == porto) & (resid["dim"] == dim)
        r_orig  = resid.loc[mask, "residual"].dropna().values
        r_innov = resid.loc[mask, "innovation"].dropna().values
        if len(r_orig) > 21:
            acf_before.append(acf(r_orig, nlags=1, fft=True)[1])
        if len(r_innov) > 21:
            acf_after.append(acf(r_innov, nlags=1, fft=True)[1])

print(f"\nACF(1) mediana: {np.median(acf_before):.3f} → {np.median(acf_after):.3f}")
print(f"|ACF(1)| > 0.15: {sum(1 for v in acf_before if abs(v)>0.15)} → "
      f"{sum(1 for v in acf_after if abs(v)>0.15)}")
print(f"|ACF(1)| > 0.30: {sum(1 for v in acf_before if abs(v)>0.30)} → "
      f"{sum(1 for v in acf_after if abs(v)>0.30)}")

# ── Sobrescrever "residual" com inovações AR(1) ──────────────
# Assim TODO o pipeline downstream (C, D, E, F, G) lê resid["residual"]
# e automaticamente recebe as inovações corrigidas.
resid["residual"] = resid["innovation"]
print(f"\n✅ resid['residual'] sobrescrito com inovações AR(1).")

# ── v8: Filtro de burn-in ─────────────────────────────────────
# As primeiras BURNIN_WEEKS semanas de cada porto×dim são modeladas
# com janela de treino insuficiente (naive fallback no walk-forward).
# Excluí-las evita contaminar a detecção com resíduos ruidosos.
resid_full = resid.copy()

resid["_date_rank"] = resid.groupby(["porto", "dim"])["date"].rank(method="dense")
resid["is_burnin"] = resid["_date_rank"] <= BURNIN_WEEKS

n_pre = len(resid)
n_cut = resid["is_burnin"].sum()
resid = resid[~resid["is_burnin"]].copy()
resid.drop(columns=["_date_rank", "is_burnin"], inplace=True)

print(f"\n── Burn-in filter ──")
print(f"  BURNIN_WEEKS = {BURNIN_WEEKS}")
print(f"  Removidos: {n_cut:,} obs ({n_cut/n_pre:.1%})")
print(f"  Restantes: {len(resid):,} obs")
print(f"  Período pós-burn-in: {resid['date'].min().date()} — {resid['date'].max().date()}")

### A6 — Diagnóstico de autocorrelação (Ljung-Box)

Teste formal de ruído branco sobre as **inovações** AR(1). Compara P1 (resíduos brutos) com as inovações corrigidas.


In [ ]:
# Célula A6 — Diagnóstico ACF (spot check) + Ljung-Box
# v7: r2 = inovações AR(1) (antes era resíduo P2 do XGBoost)

# ACF visual para 3 portos
for porto in PORTOS[:3]:
    for dim in ["atracacoes"]:
        r1 = resid_p1[(resid_p1["porto"] == porto) & (resid_p1["dim"] == dim)]["residual"].dropna().values
        r2 = resid[(resid["porto"] == porto) & (resid["dim"] == dim)]["innovation"].dropna().values
        if len(r1) < 21 or len(r2) < 21:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        plot_acf(r1, lags=20, ax=axes[0], title=f"{porto} P1 residual")
        plot_acf(r2, lags=20, ax=axes[1], title=f"{porto} AR(1) inovação")
        plt.tight_layout(); plt.show()

        lb_p1 = acorr_ljungbox(r1, lags=[10], return_df=True)["lb_pvalue"].values[0]
        lb_p2 = acorr_ljungbox(r2, lags=[10], return_df=True)["lb_pvalue"].values[0]
        print(f"  {porto} — Ljung-Box P1: p={lb_p1:.4f} → AR(1) inovação: p={lb_p2:.4f} "
              f"({'✅' if lb_p2 > 0.05 else '⚠️ ainda significante'})")

# Ljung-Box completo
print("\n── Ljung-Box completo (atracacoes – inovações) ──")
ljung_results = []
for porto in PORTOS:
    r2 = resid[(resid["porto"]==porto) & (resid["dim"]=="atracacoes")]["innovation"].dropna().values
    if len(r2) < 21: continue
    lb = acorr_ljungbox(r2, lags=[10], return_df=True)["lb_pvalue"].values[0]
    ljung_results.append({"porto": porto, "lb_p": lb})

ljung_df = pd.DataFrame(ljung_results)
n_pass = (ljung_df["lb_p"] > 0.05).sum()
print(f"Ljung-Box inovações: {n_pass}/{len(ljung_df)} portos com p > 0.05")
print(ljung_df.to_string(index=False))


## ═══ SEÇÃO B: DIAGNÓSTICO DE RESÍDUOS E JUSTIFICATIVA AR(1) ═══

O Ljung-Box rejeita H₀ (ruído branco) para os resíduos P1 em todos os 20 portos. Com ~570 observações, mesmo ACFs pequenas produzem p ≈ 0.

Esta seção documenta:
1. **Magnitude da autocorrelação residual** → ACF(1) por porto × dimensão nos resíduos P1
2. **Efeito do filtro AR(1)** → comparação ACF(1) antes/depois nas inovações

A correção AR(1) (célula A5) reduz a ACF(1) mediana de ~0.37 para ~−0.06, justificando sua inclusão permanente no pipeline.


### B1 — Magnitude da autocorrelação residual (ACF(1))

Calculamos a autocorrelação lag-1 para cada porto × dimensão nos resíduos P1 e P2. A questão-chave não é "o p-valor é significante?" (sempre será com N grande), mas "qual é o tamanho da ACF(1)?"

- |ACF(1)| < 0.10: negligível — Ljung-Box rejeita por poder estatístico, não por estrutura relevante
- |ACF(1)| 0.10–0.20: pequena — anomalias de grande magnitude ainda se destacam
- |ACF(1)| > 0.20: substancial — necessário teste de robustez


In [ ]:
# ══════════════════════════════════════════════════════════════
# DIAGNÓSTICO: Magnitude da autocorrelação residual (P1 vs P2)
# ══════════════════════════════════════════════════════════════
# O Ljung-Box rejeita para todos os portos, mas o p-valor não
# mede o TAMANHO da autocorrelação. Esta célula calcula:
#   1. ACF(1) por porto×dim (P1 e P2) → quão correlacionados são vizinhos
#   2. Melhoria P1→P2 → o two-pass está funcionando?
#   3. Ljung-Box statistic (não só p-valor) → magnitude absoluta
#   4. Resumo visual e tabela para artigo/defesa

from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

# ── 1. ACF(1) por porto × dim ──────────────────────────────
rows = []
for porto in PORTOS:
    for dim in DIMS:
        # Resíduos P1
        r1 = resid_p1[(resid_p1["porto"] == porto) & (resid_p1["dim"] == dim)]["residual"].dropna().values
        # Resíduos P2 (final)
        r2 = resid[(resid["porto"] == porto) & (resid["dim"] == dim)]["residual"].dropna().values

        if len(r1) < 21 or len(r2) < 21:
            continue

        acf1_p1 = acf(r1, nlags=5, fft=True)[1]  # lag-1
        acf1_p2 = acf(r2, nlags=5, fft=True)[1]
        acf2_p2 = acf(r2, nlags=5, fft=True)[2]   # lag-2 também

        # Ljung-Box statistic (não só p-valor)
        lb_p1 = acorr_ljungbox(r1, lags=[10], return_df=True)
        lb_p2 = acorr_ljungbox(r2, lags=[10], return_df=True)

        rows.append({
            "porto": porto,
            "dim": dim,
            "n_obs": len(r2),
            "acf1_p1": acf1_p1,
            "acf1_p2": acf1_p2,
            "acf2_p2": acf2_p2,
            "melhoria_acf1": acf1_p1 - acf1_p2,           # positivo = P2 melhorou
            "melhoria_pct": (1 - abs(acf1_p2)/abs(acf1_p1)) * 100 if acf1_p1 != 0 else np.nan,
            "lb_stat_p1": lb_p1["lb_stat"].values[0],
            "lb_stat_p2": lb_p2["lb_stat"].values[0],
            "lb_p_p2": lb_p2["lb_pvalue"].values[0],
        })

diag = pd.DataFrame(rows)

# ── 2. Resumo geral ────────────────────────────────────────
print("=" * 70)
print("DIAGNÓSTICO DE AUTOCORRELAÇÃO RESIDUAL")
print("=" * 70)

print("\n── ACF(1) dos resíduos P2 (final) ──")
print(f"  Mediana:  {diag['acf1_p2'].median():.3f}")
print(f"  Média:    {diag['acf1_p2'].mean():.3f}")
print(f"  Min:      {diag['acf1_p2'].min():.3f}")
print(f"  Max:      {diag['acf1_p2'].max():.3f}")
print(f"  |ACF(1)| < 0.10:  {(diag['acf1_p2'].abs() < 0.10).sum()}/{len(diag)}")
print(f"  |ACF(1)| < 0.15:  {(diag['acf1_p2'].abs() < 0.15).sum()}/{len(diag)}")
print(f"  |ACF(1)| < 0.20:  {(diag['acf1_p2'].abs() < 0.20).sum()}/{len(diag)}")
print(f"  |ACF(1)| > 0.30:  {(diag['acf1_p2'].abs() > 0.30).sum()}/{len(diag)}  ← preocupante")

print(f"\n── Melhoria P1 → P2 ──")
print(f"  Mediana melhoria |ACF(1)|: {diag['melhoria_pct'].median():.1f}%")
print(f"  Portos×dims que melhoraram: {(diag['melhoria_acf1'] > 0).sum()}/{len(diag)}")

print(f"\n── Ljung-Box statistic (não p-valor) ──")
print(f"  Mediana LB stat P1: {diag['lb_stat_p1'].median():.1f}")
print(f"  Mediana LB stat P2: {diag['lb_stat_p2'].median():.1f}")
print(f"  Redução mediana:    {(1 - diag['lb_stat_p2'].median()/diag['lb_stat_p1'].median())*100:.0f}%")

# ── 3. Por dimensão ────────────────────────────────────────
print("\n── ACF(1) mediana por dimensão ──")
for dim in DIMS:
    sub = diag[diag["dim"] == dim]
    print(f"  {dim:20s}: P1={sub['acf1_p1'].median():.3f} → P2={sub['acf1_p2'].median():.3f}"
          f"  (melhoria {sub['melhoria_pct'].median():.0f}%)")

# ── 4. Piores portos (ACF(1) > 0.20) ──────────────────────
print("\n── Portos com |ACF(1)| > 0.20 no P2 ──")
worst = diag[diag["acf1_p2"].abs() > 0.20].sort_values("acf1_p2", ascending=False, key=abs)
if len(worst) == 0:
    print("  Nenhum! ✅")
else:
    print(worst[["porto", "dim", "acf1_p1", "acf1_p2", "melhoria_pct", "lb_stat_p2"]].to_string(index=False))

# ── 5. Tabela completa (para artigo) ──────────────────────
print("\n── Tabela completa: mediana por porto ──")
summary = diag.groupby("porto").agg(
    acf1_p1_med=("acf1_p1", "median"),
    acf1_p2_med=("acf1_p2", "median"),
    melhoria_med=("melhoria_pct", "median"),
    lb_stat_p2_med=("lb_stat_p2", "median"),
    max_abs_acf1=("acf1_p2", lambda x: x.abs().max()),
).round(3).sort_values("max_abs_acf1", ascending=False)
print(summary.to_string())

# ── 6. Visualização ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma ACF(1) P2
axes[0].hist(diag["acf1_p2"], bins=30, color="#4fc3f7", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="white", ls="--", lw=1)
axes[0].axvline(0.15, color="red", ls="--", lw=1, label="|ACF|=0.15")
axes[0].axvline(-0.15, color="red", ls="--", lw=1)
axes[0].set_xlabel("ACF(1) resíduos P2")
axes[0].set_ylabel("Contagem (porto×dim)")
axes[0].set_title("Distribuição ACF(1) — Resíduos P2")
axes[0].legend()

# Scatter P1 vs P2
axes[1].scatter(diag["acf1_p1"], diag["acf1_p2"], alpha=0.5, s=30, c="#4fc3f7")
lim = max(diag["acf1_p1"].abs().max(), diag["acf1_p2"].abs().max()) * 1.1
axes[1].plot([-lim, lim], [-lim, lim], "r--", lw=1, label="P1=P2 (sem melhoria)")
axes[1].set_xlabel("ACF(1) — Passe 1")
axes[1].set_ylabel("ACF(1) — Passe 2")
axes[1].set_title("Melhoria P1 → P2")
axes[1].legend()

# Box plot por dimensão
diag.boxplot(column="acf1_p2", by="dim", ax=axes[2])
axes[2].axhline(0, color="white", ls="--", lw=1)
axes[2].axhline(0.15, color="red", ls="--", lw=1)
axes[2].axhline(-0.15, color="red", ls="--", lw=1)
axes[2].set_title("ACF(1) P2 por dimensão")
axes[2].set_xlabel("")
plt.suptitle("")

plt.tight_layout()
plt.savefig(OUT / "diagnostico_acf.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 7. Interpretação automática ────────────────────────────
med = diag["acf1_p2"].abs().median()
pct_above_015 = (diag["acf1_p2"].abs() > 0.15).mean() * 100
pct_above_030 = (diag["acf1_p2"].abs() > 0.30).mean() * 100

print("\n" + "=" * 70)
print("INTERPRETAÇÃO")
print("=" * 70)
if med < 0.10:
    print(f"✅ ACF(1) mediana = {med:.3f} (< 0.10)")
    print("   Autocorrelação residual é NEGLIGÍVEL.")
    print("   Ljung-Box rejeita por poder estatístico alto (N grande), não por")
    print("   estrutura temporal relevante. Detecção de anomalias NÃO é comprometida.")
elif med < 0.20:
    print(f"⚠️ ACF(1) mediana = {med:.3f} (0.10–0.20)")
    print("   Autocorrelação residual é PEQUENA mas presente.")
    print("   Anomalias de grande magnitude (>2.5σ) ainda se destacam,")
    print("   mas perto dos thresholds pode haver falsos positivos clusters.")
    print("   Documentar como limitação.")
else:
    print(f"🔴 ACF(1) mediana = {med:.3f} (> 0.20)")
    print("   Autocorrelação residual é SUBSTANCIAL.")
    print("   O modelo não captura estrutura temporal significativa.")
    print("   Resultados de detecção devem ser interpretados com cautela.")

print(f"\n  {pct_above_015:.0f}% dos porto×dim com |ACF(1)| > 0.15")
print(f"  {pct_above_030:.0f}% dos porto×dim com |ACF(1)| > 0.30")


### B2 — Comparação ACF(1): resíduos P1 vs inovações AR(1)

Em v5 e v6, o XGBoost com lags de resíduo (Passe 2) melhorava a ACF(1) em apenas ~0.1%. O filtro AR(1) reduz a ACF(1) de 0.37 para −0.06 com custo computacional negligível. Esta célula mostra o diagnóstico completo usando os `acf_before`/`acf_after` calculados em A5.


In [ ]:
# Célula B2 — Diagnóstico comparativo: ACF(1) resíduos P1 vs inovações AR(1)
# Usa acf_before / acf_after calculados em A5.
# Em v5/v6, o Passe 2 XGBoost melhorava ACF(1) em 0.1%; AR(1) a reduz de 0.37 → −0.06.

print("=" * 60)
print("ACF(1): resíduos P1 vs inovações AR(1)")
print("=" * 60)
print(f"  {'Métrica':<30s} {'P1 resid':>10s} {'AR(1) innov':>12s}")
print(f"  {'-'*52}")
print(f"  {'Mediana ACF(1)':<30s} {np.median(acf_before):>10.3f} {np.median(acf_after):>12.3f}")
print(f"  {'Média ACF(1)':<30s} {np.mean(acf_before):>10.3f} {np.mean(acf_after):>12.3f}")
print(f"  {'|ACF(1)| > 0.15':<30s} {sum(1 for v in acf_before if abs(v)>0.15):>10d} "
      f"{sum(1 for v in acf_after if abs(v)>0.15):>12d}")
print(f"  {'|ACF(1)| > 0.20':<30s} {sum(1 for v in acf_before if abs(v)>0.20):>10d} "
      f"{sum(1 for v in acf_after if abs(v)>0.20):>12d}")
print(f"  {'|ACF(1)| > 0.30':<30s} {sum(1 for v in acf_before if abs(v)>0.30):>10d} "
      f"{sum(1 for v in acf_after if abs(v)>0.30):>12d}")

# Histograma comparativo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(acf_before, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(np.median(acf_before), color='red', linestyle='--',
                label=f'mediana={np.median(acf_before):.3f}')
axes[0].set_title('ACF(1) — Resíduos P1'); axes[0].legend()
axes[1].hist(acf_after, bins=20, color='seagreen', edgecolor='white')
axes[1].axvline(np.median(acf_after), color='red', linestyle='--',
                label=f'mediana={np.median(acf_after):.3f}')
axes[1].set_title('ACF(1) — Inovações AR(1)'); axes[1].legend()
plt.suptitle('Efeito do filtro AR(1) sobre autocorrelação residual')
plt.tight_layout(); plt.show()
print("\n✅ AR(1) reduz ACF(1) substancialmente. Inovações ≈ ruído branco.")


### Conclusão da Seção B

O filtro AR(1) (célula A5) reduz a ACF(1) mediana de ~0.37 para ~−0.06, aproximando as inovações de ruído branco. A detecção ensemble (Seção C) opera sobre essas inovações, garantindo que desvios detectados não sejam artefatos da autocorrelação.

Documentação para o artigo:
> "Os resíduos do modelo XGBoost (own-only features) apresentam autocorrelação residual (ACF(1) mediana = 0.37). Aplicamos filtro AR(1) por porto × dimensão para calcular inovações $\varepsilon_t = r_t - \hat{\rho} \cdot r_{t-1}$, que apresentam ACF(1) mediana de −0.06 (≈ ruído branco). A detecção de anomalias opera sobre as inovações."

**Nota:** Em versões anteriores (v5, v6), o Passe 2 do XGBoost com lags de resíduo melhorava a ACF(1) em apenas 0.1%. O AR(1) é uma solução superior por ser analiticamente exata para dependência serial de ordem 1.

---


## ═══ SEÇÃO C: DETECÇÃO DE ANOMALIAS ═══

Ensemble de 3 métodos sobre as **inovações AR(1)** ($\varepsilon_t$), com threshold adaptativo por porto × dimensão:
1. **MAD** — Median Absolute Deviation (robusto a outliers)
2. **STL** — Decomposição sazonal residual (captura sazonalidade remanescente)
3. **Isolation Forest** — Detecção multivariada (inovação, diff, desvio de MM30)

**Regra de consenso:** anomalia confirmada se ≥2 métodos concordam.

**Threshold adaptativo:** portos estáveis (baixo CV) → threshold menor (mais sensível); portos voláteis → threshold maior (menos falsos positivos). Floor mínimo uniforme k_min=2.0 para todas as dimensões.

**Burn-in:** As primeiras 78 semanas (~1.5 anos) de cada série porto×dim são excluídas da detecção para permitir estabilização do modelo.


In [ ]:
# Célula C1 — Ensemble de anomalias com THRESHOLD ADAPTATIVO por porto×dim
# v8: opera sobre inovações AR(1) pós burn-in
# Portos estáveis (baixo CV) → threshold menor → mais sensível
# Portos voláteis (alto CV) → threshold maior → menos falsos positivos

def detect_mad(r, k=MAD_K):
    med = np.nanmedian(r); mad = np.nanmedian(np.abs(r - med))
    return np.abs(r - med) > k * 1.4826 * mad

def detect_stl_r(r, period=STL_PERIOD, threshold=STL_RESID_ZSCORE):
    s = pd.Series(r).interpolate().bfill().ffill()
    if s.notna().sum() < 2 * period:
        return np.full(len(r), False)
    try:
        sr = STL(s, period=period, robust=True).fit().resid.values
        z = np.abs(sr - np.nanmean(sr)) / max(np.nanstd(sr), 1e-10)
        return z > threshold
    except:
        return np.full(len(r), False)

def detect_if(r, contamination=IFOREST_CONTAMINATION):
    s = pd.Series(r)
    X = pd.DataFrame({
        "r": s, "rd": s.diff(),
        "rd2": s - s.rolling(30, min_periods=10).mean()
    }).fillna(0).values
    return IsolationForest(contamination=contamination, random_state=42,
                            n_estimators=200).fit_predict(X) == -1

# ── Calcular CV por porto×dim para threshold adaptativo ──
# v7: usa inovações AR(1) para CV
cv_by_port = {}
for porto in PORTOS:
    for dim in DIMS:
        mask = (resid["porto"] == porto) & (resid["dim"] == dim)
        r = resid.loc[mask, "innovation"].dropna().values
        if len(r) > 52:
            cv_val = np.std(r) / max(abs(np.mean(r)), 1e-10)
            cv_by_port[(porto, dim)] = cv_val

cv_values = list(cv_by_port.values())
cv_mediano = np.median(cv_values) if cv_values else 1.0

print(f"CV das inovações (mediana={cv_mediano:.3f}):")
print(f"Floors por dimensão: {ADAPTIVE_K_FLOORS}")

# ── Detecção com threshold adaptativo ──
resid["a_ens"] = False
anomaly_stats = []

for porto in PORTOS:
    for dim in DIMS:
        mask = (resid["porto"] == porto) & (resid["dim"] == dim)
        r = resid.loc[mask, "innovation"].values

        if len(r) < 104 or np.isnan(r).sum() / len(r) > 0.5:
            continue

        # Threshold adaptativo COM FLOOR POR DIMENSÃO
        cv_val = cv_by_port.get((porto, dim), cv_mediano)
        ratio = cv_val / cv_mediano if cv_mediano > 0 else 1.0
        k_floor = ADAPTIVE_K_FLOORS.get(dim, 2.0)     # ← FLOOR ESPECÍFICO
        k_porto = np.clip(MAD_K * ratio, k_floor, ADAPTIVE_K_MAX)

        # ── Log k=NaN com fallback documentado ──
        if np.isnan(k_porto):
            print(f"  ⚠️ k=NaN: {porto}×{dim} (cv={cv_val:.4f}, ratio={ratio:.4f}) → fallback k_floor={k_floor:.2f}")
            k_porto = k_floor

        # Detectar com k adaptado
        f1 = detect_mad(r, k=k_porto)
        f2 = detect_stl_r(r, threshold=k_porto)
        f3 = detect_if(r)

        agree = f1.astype(int) + f2.astype(int) + f3.astype(int)
        resid.loc[mask, "a_ens"] = agree >= ENSEMBLE_MIN_AGREEMENT

        n_anom = (agree >= ENSEMBLE_MIN_AGREEMENT).sum()
        anomaly_stats.append({
            "porto": porto, "dim": dim,
            "k_floor": k_floor, "k_adaptado": k_porto, "cv_ratio": ratio,
            "n_anomalias": n_anom, "pct": n_anom / max(len(r), 1)
        })

# Resumo
astat = pd.DataFrame(anomaly_stats)
n_anom = resid["a_ens"].sum()
print(f"\nAnomalias ensemble (sobre inovações): {n_anom:.0f} / {resid['a_ens'].notna().sum()} "
      f"({n_anom / resid['a_ens'].notna().sum():.1%})")

print(f"\nPor dimensão:")
for dim in DIMS:
    n = resid[(resid["dim"] == dim) & (resid["a_ens"] == True)].shape[0]
    k_range = astat[astat["dim"]==dim]["k_adaptado"]
    print(f"  {dim:20s}: {n:4d} anomalias  (k: {k_range.min():.2f}–{k_range.max():.2f})")

print(f"\nPor porto (threshold adaptativo, k_base={MAD_K}):")
print(f"  {'Porto':25s} {'k_atr':>6s} {'k_ton':>6s} {'k_t1':>6s} {'k_tat':>6s} {'total':>6s}")
for porto in PORTOS:
    ks = {}
    for dim in DIMS:
        row = astat[(astat["porto"]==porto) & (astat["dim"]==dim)]
        ks[dim] = row["k_adaptado"].values[0] if len(row) > 0 else np.nan
    total = astat[astat["porto"]==porto]["n_anomalias"].sum()
    print(f"  {porto:25s} {ks.get('atracacoes',0):6.2f} {ks.get('tonelagem_exp',0):6.2f} "
          f"{ks.get('t1_mediano',0):6.2f} {ks.get('tatracado_mediano',0):6.2f} {total:6.0f}")


### C1b — Near-misses (quase-anomalias)

Observações com MAD-distance entre 0.8×k e k — logo abaixo do limiar de detecção. Capturam disrupções de sinal mais fraco, como a Greve da Receita Federal, que podem não atingir o threshold adaptativo.

In [ ]:
# Célula C1b — Exportar near-misses (quase-anomalias: 0.8×k ≤ MAD-dist < k)
# Captura disrupções abaixo do limiar mas próximas — útil para eventos
# com sinal difuso que não atingem o threshold adaptativo.

near_miss_rows = []
for porto in PORTOS:
    for dim in DIMS:
        mask = (resid["porto"] == porto) & (resid["dim"] == dim)
        r = resid.loc[mask, "innovation"].values
        dates_nm = resid.loc[mask, "date"].values

        if len(r) < 104:
            continue

        # MAD distance
        med = np.nanmedian(r)
        mad = np.nanmedian(np.abs(r - med))
        mad_scaled = 1.4826 * mad
        if mad_scaled < 1e-10:
            continue

        mad_dist = np.abs(r - med) / mad_scaled

        # Threshold deste porto×dim
        row_stat = astat[(astat["porto"] == porto) & (astat["dim"] == dim)]
        if len(row_stat) == 0:
            continue
        k_porto = row_stat["k_adaptado"].values[0]

        # Near-misses: 0.8×k ≤ MAD-dist < k  AND  não é anomalia
        is_anomaly = resid.loc[mask, "a_ens"].values
        nm_mask = (mad_dist >= 0.8 * k_porto) & (mad_dist < k_porto) & (~is_anomaly)

        for idx in np.where(nm_mask)[0]:
            near_miss_rows.append({
                "porto": porto, "dim": dim,
                "date": dates_nm[idx],
                "mad_dist": round(float(mad_dist[idx]), 3),
                "k_adaptado": round(float(k_porto), 3),
                "pct_of_k": round(float(mad_dist[idx] / k_porto), 3),
                "innovation": round(float(r[idx]), 4),
            })

near_misses = pd.DataFrame(near_miss_rows)
print(f"Near-misses (0.8k ≤ MAD-dist < k): {len(near_misses)}")

if len(near_misses) > 0:
    print(f"\nPor dimensão:")
    print(near_misses.groupby("dim").size().to_string())

    print(f"\nPor porto (top 10):")
    print(near_misses.groupby("porto").size().sort_values(ascending=False).head(10).to_string())

    # Salvar
    near_misses.to_parquet(OUT / "near_misses.parquet", index=False)
    near_misses.to_csv(OUT / "near_misses.csv", index=False)
    print(f"\n✅ near_misses exportados: {len(near_misses)} linhas")

    # Verificação: Greve da Receita Federal
    greve_rf = near_misses[
        (near_misses["date"] >= np.datetime64("2024-12-01")) &
        (near_misses["date"] <= np.datetime64("2025-01-15"))
    ]
    if len(greve_rf) > 0:
        print(f"\n🔍 Near-misses no período da Greve da Receita Federal:")
        print(greve_rf[["porto", "dim", "date", "mad_dist", "k_adaptado", "pct_of_k"]]
              .to_string(index=False))
    else:
        print(f"\nNenhum near-miss no período da Greve da Receita Federal")

    # Verificação: Enchentes RS
    enchentes = near_misses[
        (near_misses["date"] >= np.datetime64("2024-04-25")) &
        (near_misses["date"] <= np.datetime64("2024-06-30")) &
        (near_misses["porto"].isin(["Rio Grande", "Porto Alegre"]))
    ]
    if len(enchentes) > 0:
        print(f"\n🔍 Near-misses Enchentes RS:")
        print(enchentes[["porto", "dim", "date", "mad_dist", "k_adaptado", "pct_of_k"]]
              .to_string(index=False))
else:
    print("Nenhum near-miss encontrado.")

### C2 — Verificação: Greve dos Caminhoneiros (mai/jun 2018)

Teste de sanidade com evento-âncora de magnitude conhecida. Verifica se o pipeline detecta anomalias nos portos mais afetados e quão próximos ficam os desvios dos thresholds.


In [ ]:
# Célula C1b — Verificação: Greve em Santos + Paranaguá + Rio Grande
print("=" * 70)
print("VERIFICAÇÃO: Greve dos Caminhoneiros (mai/jun 2018)")
print("=" * 70)

greve_start = pd.Timestamp("2018-05-21")
greve_end = pd.Timestamp("2018-06-10")

# Contexto expandido (±2 semanas) para capturar efeitos defasados
greve_ctx_start = greve_start - pd.Timedelta(weeks=2)
greve_ctx_end = greve_end + pd.Timedelta(weeks=2)

for porto in ["Santos", "Paranaguá", "Rio Grande", "Itaguaí", "Suape"]:
    print(f"\n  {porto}:")
    for dim in DIMS:
        mask = (resid["porto"] == porto) & (resid["dim"] == dim)
        sub = resid.loc[mask].sort_values("date")

        # Resíduos no período da greve
        greve_mask = (sub["date"] >= greve_start) & (sub["date"] <= greve_end)
        greve_r = sub.loc[greve_mask]

        # Contexto expandido
        ctx_mask = (sub["date"] >= greve_ctx_start) & (sub["date"] <= greve_ctx_end)
        ctx_r = sub.loc[ctx_mask]

        if greve_r.empty:
            print(f"    {dim:20s}: sem dados no período")
            continue

        n_nan = greve_r["residual"].isna().sum()
        n_anom_strict = greve_r["a_ens"].sum() if "a_ens" in greve_r else 0
        n_anom_ctx = ctx_r["a_ens"].sum() if "a_ens" in ctx_r else 0
        residuals = greve_r["residual"].dropna().values

        # Threshold adaptativo deste porto×dim
        k_porto = ADAPTIVE_K_MAX  # fallback
        row_stat = astat[(astat["porto"]==porto) & (astat["dim"]==dim)]
        if len(row_stat) > 0:
            k_porto = row_stat["k_adaptado"].values[0]

        if len(residuals) > 0:
            r_all = sub["residual"].dropna().values
            med = np.nanmedian(r_all)
            mad = np.nanmedian(np.abs(r_all - med))
            mad_scaled = 1.4826 * mad
            mad_dist = [abs(v - med) / max(mad_scaled, 1e-10) for v in residuals]
            max_dist = max(mad_dist)

            # Diagnóstico
            if n_anom_strict > 0:
                status = "✅ DETECTADO"
            elif n_anom_ctx > 0:
                status = "🟡 DETECTADO ±2sem"
            elif max_dist > k_porto * 0.7:
                status = f"⚠️ NEAR-MISS ({max_dist:.1f}σ vs k={k_porto:.1f})"
            else:
                status = f"❌ LONGE ({max_dist:.1f}σ vs k={k_porto:.1f})"

            print(f"    {dim:20s}: MAD-dist max={max_dist:.1f}σ, k={k_porto:.2f}, "
                  f"anom={n_anom_strict}(+{n_anom_ctx-n_anom_strict} ctx)  {status}")
        else:
            print(f"    {dim:20s}: NaN={n_nan}, todos os resíduos NaN")

    # Resumo do porto
    all_anom = resid[(resid["porto"]==porto) & 
                     (resid["date"]>=greve_start) & (resid["date"]<=greve_end) &
                     (resid["a_ens"]==True)]
    print(f"    {'─'*50}")
    print(f"    Total anomalias no período: {len(all_anom)} "
          f"(dims: {sorted(all_anom['dim'].unique().tolist()) if len(all_anom) > 0 else 'nenhuma'})")

### C3 — Padrões multidimensionais

Cada anomalia (porto × semana) pode afetar 1-4 dimensões. Classificamos o padrão combinado para caracterizar o tipo de choque operacional.


In [ ]:
# Célula C2 — Padrões multidimensionais
# v8: tatracado_mediano no lugar de pct_paralisacao

def classify_pattern(dims_set):
    if dims_set >= set(DIMS):
        return "severo"             # todas as 4 dims
    if {"atracacoes", "tonelagem_exp"} <= dims_set:
        return "demanda"            # choque de demanda
    if {"t1_mediano", "tatracado_mediano"} <= dims_set:
        return "gargalo"            # gargalo operacional
    if "t1_mediano" in dims_set and "atracacoes" not in dims_set:
        return "operacional"        # T1 alto sem queda de atracações
    if dims_set == {"tatracado_mediano"}:
        return "berco"
    if len(dims_set) >= 2:
        return "misto"
    return f"isolado_{list(dims_set)[0]}"

ens = resid[resid["a_ens"] == True].copy()
patterns = (ens.groupby(["porto", "date"])["dim"]
    .agg(lambda x: classify_pattern(set(x)))
    .reset_index(name="padrao"))

print("Padrões multidimensionais:")
print(patterns["padrao"].value_counts())


## ═══ SEÇÃO D: CLASSIFICAÇÃO (Dual Score) ═══

Cada anomalia detectada é classificada em 3 níveis por dois scores paralelos:

| Score | Mede | Fonte |
|---|---|---|
| **Score A** | Co-ocorrência entre portos BR na mesma dimensão (±7 semanas) | Contagem de portos afetados |
| **Score B** | Índice de disrupção global PortWatch (|z-score| na janela ±2 semanas) | PortWatch (2019+) |

**Regra de classificação:**
- **Global:** Score B ≥ tb (disrupção global detectada), independente de Score A
- **Nacional:** Score A ≥ ta E Score B < tb (muitos portos BR afetados, sem trigger global)
- **Isolado:** Score A < ta E Score B < tb (poucos portos, sem trigger global)

**Nota sobre período pré-2019:** O índice global PortWatch começa em 2019. Anomalias anteriores têm Score B = 0 por construção, sendo classificadas como "nacional" ou "isolado" — nunca "global". Isso é uma limitação estrutural documentada.


In [ ]:
# Célula D1 — Score A: co-ocorrência same-dim
# Fix 1: usa DIMS (todas as 4)
# v7: passa 'innovation' além de 'residual' para fingerprint (E1)

def compute_score_a(ens, window=COOC_WINDOW_WEEKS, same_dim=COOC_SAME_DIM):
    results = []
    for dim in DIMS:
        ens_d = ens[ens["dim"] == dim] if same_dim else ens
        if len(ens_d) == 0:
            continue
        lookup = ens_d.groupby("date")["porto"].apply(set).to_dict()

        for _, row in ens[ens["dim"] == dim].iterrows():
            portos_w = set()
            for dw in range(-window, window + 1):
                d = row["date"] + pd.Timedelta(weeks=dw)
                if d in lookup:
                    portos_w |= lookup[d]
            results.append({
                "porto": row["porto"], "date": row["date"], "dim": dim,
                "residual": row["residual"],
                "innovation": row.get("innovation", row["residual"]),
                "score_a": len(portos_w),
                "portos_co": sorted(portos_w - {row["porto"]}),
            })

    return pd.DataFrame(results)

scores = compute_score_a(ens)
print(f"Score A calculado: {len(scores)}")
print(f"Distribuição Score A:\n{scores['score_a'].value_counts().sort_index()}")
print(f"\nPor dimensão:")
for dim in DIMS:
    n = scores[scores["dim"] == dim].shape[0]
    print(f"  {dim}: {n}")


In [ ]:
# Célula D2 — Score B: índice global
# TRACER v3.1: |gi_z| na semana da anomalia, janela ±2 semanas.

def add_score_b(scores_df, gi, window=COOC_GI_WINDOW_WEEKS):
    gi_z = gi.set_index("date")["gi_z"] if "gi_z" in gi.columns else gi.set_index("date").iloc[:, 0]

    score_b = []
    for _, row in scores_df.iterrows():
        vals = []
        for dw in range(-window, window + 1):
            d = row["date"] + pd.Timedelta(weeks=dw)
            if d in gi_z.index:
                vals.append(abs(gi_z[d]))
        score_b.append(max(vals) if vals else 0.0)

    scores_df = scores_df.copy()
    scores_df["score_b"] = score_b
    return scores_df

scores = add_score_b(scores, gi)
print(f"Score B adicionado. Estatísticas:")
print(scores[["score_a", "score_b"]].describe().round(2))

In [ ]:
# Célula D3 — Grid search: recall sobre eventos conhecidos
# Fix 2: Inclui greves (enchentes já estão)

grid_ta = [3, 4, 5, 6]
grid_tb = [0.8, 1.0, 1.2, 1.5]

eval_events = [
    # Enchentes RS out/24 (porto Rio Grande)
    {"date": "2024-10-21", "portos": ["Rio Grande"], "tipo": "global"},
    {"date": "2024-10-28", "portos": ["Rio Grande"], "tipo": "global"},
    # Greve caminhoneiros mai/2018
    {"date": "2018-05-21", "portos": ["Santos", "Paranaguá", "Rio Grande"], "tipo": "nacional"},
    {"date": "2018-05-28", "portos": ["Santos", "Paranaguá", "Rio Grande"], "tipo": "nacional"},
    # Enchentes RS mai/24 - imediato
    {"date": "2024-05-06", "portos": ["Rio Grande"], "tipo": "global"},
    {"date": "2024-05-13", "portos": ["Rio Grande"], "tipo": "global"},
    {"date": "2024-05-20", "portos": ["Rio Grande"], "tipo": "global"},
]

n_events = len(eval_events)
print(f"Total de eventos conhecidos para avaliação: {n_events}")

def classify_tri(df, ta, tb):
    """Classificação tripartite: isolado / nacional / global"""
    conds = [
        df["score_b"] >= tb,                          # global se score_b alto
        (df["score_a"] >= ta) & (df["score_b"] < tb), # nacional se score_a alto e score_b baixo
    ]
    choices = ["global", "nacional"]
    return np.select(conds, choices, default="isolado")

results_grid = []
for ta in grid_ta:
    for tb in grid_tb:
        c = classify_tri(scores, ta, tb)
        n_global = (c == "global").sum()
        n_nacional = (c == "nacional").sum()
        n_isolado = (c == "isolado").sum()
        
        hits = []
        for ev in eval_events:
            ev_date = pd.to_datetime(ev["date"])
            portos_ev = ev["portos"]
            expected = ev["tipo"]
            
            mask = (scores["date"] == ev_date) & (scores["porto"].isin(portos_ev))
            row = scores[mask]
            if len(row) == 0:
                continue
            
            sub_cls = c[mask.values]
            if len(sub_cls) > 0:
                from collections import Counter
                mode_cls = Counter(sub_cls).most_common(1)[0][0]

                if expected == "global" and mode_cls == "global":
                    hits.append(1)
                elif expected == "nacional" and mode_cls in ["nacional", "global"]:
                    hits.append(1)
                else:
                    hits.append(0)
        
        recall = sum(hits) / n_events if n_events > 0 else 0
        results_grid.append({
            "ta": ta, "tb": tb, 
            "n_global": n_global, "n_nacional": n_nacional, "n_isolado": n_isolado,
            "pct_global": round(100 * n_global / len(c), 1) if len(c) > 0 else 0,
            "hits": hits, "total_events": n_events, "recall": round(recall, 2)
        })

grid_df = pd.DataFrame(results_grid).sort_values(["recall", "n_global"], ascending=[False, True]).reset_index(drop=True)
grid_df["hits"] = grid_df["hits"].apply(lambda x: sum(x))
print("Grid search results:")
print(grid_df.to_string(index=False))

best = grid_df.iloc[0]
# Decisão metodológica: ta=3 (limite conservador), tb=0.8 (percentil 79)
# Grid search indica ta=4 com +1 hit, mas desempenho estável em [3,5]
BEST_TA = 3
BEST_TB = 0.8
print(f"\nGrid search indicou: ta={int(best['ta'])}, tb={float(best['tb'])}, recall={best['recall']}")
print(f"Forçado (decisão metodológica): ta={BEST_TA}, tb={BEST_TB}")

In [ ]:
# Célula D4 — Aplicar melhor combinação e distribuição final
# Fix 3: 3 níveis (isolado / nacional / global)
scores["classificacao"] = classify_tri(scores, BEST_TA, BEST_TB)

print("Distribuição final de classificação (3 níveis):")
print(scores["classificacao"].value_counts())
print(f"\nPor dimensão:")
print(scores.groupby(["dim", "classificacao"]).size().unstack(fill_value=0))
print(f"\nPor porto:")
print(scores.groupby(["porto", "classificacao"]).size().unstack(fill_value=0))

In [ ]:
# Célula D5 — Validação com eventos conhecidos (KNOWN_EVENTS do config.py)
# KNOWN_EVENTS já importado via "from config import *" na célula 0

print("=" * 60)
print("VALIDAÇÃO COM EVENTOS CONHECIDOS (3 níveis)")
print("=" * 60)

for ev in KNOWN_EVENTS:
    if "start" not in ev or "end" not in ev:
        continue
    mask = (
        (scores["date"] >= pd.Timestamp(ev["start"]))
        & (scores["date"] <= pd.Timestamp(ev["end"]))
    )
    if "portos_foco" in ev and ev["portos_foco"]:
        mask = mask & scores["porto"].isin(ev["portos_foco"])

    sub = scores[mask]
    if len(sub) == 0:
        status = "SEM ANOMALIAS DETECTADAS"
        cls = "-"
    else:
        cls = sub["classificacao"].mode().iloc[0]
        expected = ev["expected"]
        status = "✅ ACERTO" if cls == expected else "❌ ERRO"

    portos_str = ", ".join(ev.get("portos_foco", ["todos"]))
    print(f"\n{ev['name']} ({ev['start']} → {ev['end']})")
    print(f"  Portos: {portos_str}")
    print(f"  Esperado: {ev['expected']:10s} | Obtido: {cls:10s} | {status}")
    if len(sub) > 0:
        print(f"  Score A médio: {sub['score_a'].mean():.1f}, Score B médio: {sub['score_b'].mean():.2f}")
        print(f"  Anomalias: {len(sub)} (dims: {sub['dim'].unique().tolist()})")

In [ ]:
# Célula D5b — Métricas complementares
# Usa KNOWN_EVENTS (config.py) em vez de eval_events

known_with_dates = [ev for ev in KNOWN_EVENTS if "start" in ev and "end" in ev]

hits = []
for ev in known_with_dates:
    mask = (
        (scores["date"] >= pd.Timestamp(ev["start"]))
        & (scores["date"] <= pd.Timestamp(ev["end"]))
    )
    if "portos_foco" in ev and ev["portos_foco"]:
        mask = mask & scores["porto"].isin(ev["portos_foco"])
    sub = scores[mask]
    if len(sub) > 0:
        cls = sub["classificacao"].mode().iloc[0]
        hits.append({"nome": ev["name"], "acerto": cls == ev["expected"],
                     "tipo_esperado": ev["expected"], "obtido": cls})

hits_df = pd.DataFrame(hits)
precision = hits_df["acerto"].mean() if len(hits_df) > 0 else 0
recall = len(hits_df) / len(known_with_dates) if known_with_dates else 0

# Taxa de alertas por classificação
total_obs = len(resid[resid["residual"].notna()])
taxa_total = len(scores) / total_obs

print(f"MÉTRICAS DE AVALIAÇÃO")
print(f"{'='*50}")
print(f"Recall:    {recall:.2f} ({len(hits_df)}/{len(known_with_dates)} eventos detectados)")
print(f"Precision: {precision:.2f} ({hits_df['acerto'].sum()}/{len(hits_df)} classificações corretas)")
print()
print(hits_df.to_string(index=False))
print()
print(f"Taxa de alertas: {taxa_total:.1%} das observações sinalizadas")
print(f"  Global:   {(scores['classificacao']=='global').sum()/total_obs:.2%}")
print(f"  Nacional: {(scores['classificacao']=='nacional').sum()/total_obs:.2%}")
print(f"  Isolado:  {(scores['classificacao']=='isolado').sum()/total_obs:.2%}")

In [ ]:
# Célula D5c — Bootstrap CIs para recall e precisão (par-nível)
# Expande os 10 eventos de referência em 135 pares porto×evento

import numpy as np

ALL_PORTOS = sorted(scores["porto"].unique())

# --- Expandir KNOWN_EVENTS em pares porto×evento ---
pair_detect = []   # 1 se detectado, 0 se não
pair_correct = []  # 1 se classificado corretamente (só detectados)

for ev in known_with_dates:
    portos = ev.get("portos_foco") or ALL_PORTOS
    for porto in portos:
        mask = (
            (scores["date"] >= pd.Timestamp(ev["start"]))
            & (scores["date"] <= pd.Timestamp(ev["end"]))
            & (scores["porto"] == porto)
        )
        sub = scores[mask]
        if len(sub) > 0:
            pair_detect.append(1)
            cls = sub["classificacao"].mode().iloc[0]
            pair_correct.append(1 if cls == ev["expected"] else 0)
        else:
            pair_detect.append(0)

pair_detect = np.array(pair_detect)
pair_correct = np.array(pair_correct)

n_total = len(pair_detect)
n_detected = pair_detect.sum()
n_correct = pair_correct.sum()

recall_pt = pair_detect.mean()
precision_pt = pair_correct.mean() if len(pair_correct) > 0 else 0
recall_cls = n_correct / n_total

print(f"Pares totais: {n_total}")
print(f"Detectados:   {n_detected}/{n_total} = {recall_pt:.1%}")
print(f"Corretos:     {n_correct}/{n_detected} = {n_correct/n_detected:.1%}")
print(f"Recall cls:   {n_correct}/{n_total} = {recall_cls:.1%}")

# --- Bootstrap ---
rng = np.random.default_rng(42)
B = 10_000

# Recall de detecção (135 pares)
recall_boot = np.empty(B)
for i in range(B):
    idx = rng.choice(n_total, size=n_total, replace=True)
    recall_boot[i] = pair_detect[idx].mean()
recall_ci = np.percentile(recall_boot, [2.5, 97.5])

# Precisão de classificação (só detectados)
prec_boot = np.empty(B)
for i in range(B):
    idx = rng.choice(len(pair_correct), size=len(pair_correct), replace=True)
    prec_boot[i] = pair_correct[idx].mean()
prec_ci = np.percentile(prec_boot, [2.5, 97.5])

# Recall da classificação (135 pares, correto/total)
# Construir array binário: 1 se detectado E correto, 0 caso contrário
pair_cls_recall = np.zeros(n_total, dtype=int)
j = 0
for i in range(n_total):
    if pair_detect[i] == 1:
        pair_cls_recall[i] = pair_correct[j]
        j += 1

recall_cls_boot = np.empty(B)
for i in range(B):
    idx = rng.choice(n_total, size=n_total, replace=True)
    recall_cls_boot[i] = pair_cls_recall[idx].mean()
recall_cls_ci = np.percentile(recall_cls_boot, [2.5, 97.5])

print(f"\n{'='*60}")
print(f"BOOTSTRAP CONFIDENCE INTERVALS (B={B:,}, seed=42)")
print(f"{'='*60}")
print(f"Recall detecção:      {recall_pt:.1%}  IC 95%: [{recall_ci[0]:.1%}, {recall_ci[1]:.1%}]")
print(f"Precisão classif.:    {n_correct/n_detected:.1%}  IC 95%: [{prec_ci[0]:.1%}, {prec_ci[1]:.1%}]")
print(f"Recall classif.:      {recall_cls:.1%}  IC 95%: [{recall_cls_ci[0]:.1%}, {recall_cls_ci[1]:.1%}]")

# --- Salvar CSV ---
ci_df = pd.DataFrame([
    {"metrica": "recall_deteccao", "valor": round(recall_pt, 3),
     "ci_lower": round(recall_ci[0], 3), "ci_upper": round(recall_ci[1], 3),
     "n": n_total, "B": B},
    {"metrica": "precisao_classificacao", "valor": round(n_correct/n_detected, 3),
     "ci_lower": round(prec_ci[0], 3), "ci_upper": round(prec_ci[1], 3),
     "n": int(n_detected), "B": B},
    {"metrica": "recall_classificacao", "valor": round(recall_cls, 3),
     "ci_lower": round(recall_cls_ci[0], 3), "ci_upper": round(recall_cls_ci[1], 3),
     "n": n_total, "B": B},
])
ci_path = OUT / "bootstrap_ci.csv"
ci_df.to_csv(ci_path, index=False)
print(f"\nSalvo em {ci_path}")

In [ ]:
# Célula D6 — Scatter plot Score A × Score B (usa KNOWN_EVENTS)

OUT = ROOT / "data" / "output"
OUT.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_cls = {"global": "red", "nacional": "darkorange", "isolado": "steelblue"}
markers_cls = {"global": "v", "nacional": "D", "isolado": "o"}

for cls, grp in scores.groupby("classificacao"):
    ax.scatter(grp["score_a"], grp["score_b"], label=cls,
               alpha=0.5, s=30, color=colors_cls.get(cls, "gray"),
               marker=markers_cls.get(cls, "o"))

ax.axvline(BEST_TA, color="gray", ls="--", lw=0.8, label=f"ta={BEST_TA}")
ax.axhline(BEST_TB, color="gray", ls=":",  lw=0.8, label=f"tb={BEST_TB}")

# Quadrantes
ax.text(BEST_TA + 0.3, BEST_TB + 0.1, "GLOBAL", fontsize=9, color="red", fontweight="bold")
ax.text(BEST_TA + 0.3, 0.05, "NACIONAL", fontsize=9, color="darkorange", fontweight="bold")
ax.text(0.5, BEST_TB + 0.1, "ISOLADO", fontsize=9, color="steelblue", fontweight="bold")

# Marcar eventos conhecidos
seen = set()
for ev in KNOWN_EVENTS:
    if "start" not in ev or "end" not in ev:
        continue
    if ev["name"] in seen:
        continue
    mask = (
        (scores["date"] >= pd.Timestamp(ev["start"]))
        & (scores["date"] <= pd.Timestamp(ev["end"]))
    )
    if "portos_foco" in ev and ev["portos_foco"]:
        mask = mask & scores["porto"].isin(ev["portos_foco"])
    sub = scores[mask]
    if len(sub) > 0:
        ax.scatter(sub["score_a"].mean(), sub["score_b"].mean(),
                   marker="*", s=200, edgecolors="black", linewidths=1,
                   color="gold", zorder=5)
        ax.annotate(ev["name"][:20], (sub["score_a"].mean(), sub["score_b"].mean()),
                    fontsize=7, ha="left", va="bottom")
        seen.add(ev["name"])

ax.set_xlabel("Score A (co-ocorrência)")
ax.set_ylabel("Score B (índice global |gi_z|)")
ax.set_title("Dual Score — Classificação de Anomalias (3 níveis)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(OUT / "scatter_dual_score.png", dpi=150)
plt.show()

## ═══ SEÇÃO E: FINGERPRINT ANTAQ E CLASSIFICADOR ENDÓGENO ═══

O fingerprint é o vetor de intensidade relativa por dimensão `[atracações, tonelagem, T1, paralisação]` para cada anomalia. Permite visualizar o "perfil" de cada tipo de disrupção.

O classificador endógeno (Decision Tree LOO) testa se o fingerprint sozinho é capaz de distinguir global/nacional/isolado — sem usar Score A ou Score B.


In [ ]:
# Célula E1 — Extrair fingerprints por anomalia
# Fix 1: usa DIMS (todas as 4)
# v7: usa abs(r["innovation"]) para intensidade

def extract_fingerprint(scores_df):
    """Agrupa anomalias por (porto, date) e constrói vetor de intensidade por dim.
    v7: intensidade = |inovação AR(1)| normalizada pelo máximo."""
    fp_rows = []
    grouped = scores_df.groupby(["porto", "date"])
    for (porto, date), grp in grouped:
        fp = {}
        for _, r in grp.iterrows():
            fp[r["dim"]] = abs(r["innovation"])
        mx = max(fp.values()) if fp else 1.0
        if mx == 0:
            mx = 1.0
        fp_norm = {k: v / mx for k, v in fp.items()}
        for d in DIMS:
            if d not in fp_norm:
                fp_norm[d] = 0.0
        fp_norm["porto"] = porto
        fp_norm["date"] = date
        fp_norm["classificacao"] = grp["classificacao"].iloc[0]
        fp_norm["score_a"] = grp["score_a"].iloc[0]
        fp_norm["score_b"] = grp["score_b"].iloc[0]
        fp_rows.append(fp_norm)
    return pd.DataFrame(fp_rows)

fingerprints = extract_fingerprint(scores)
print(f"Fingerprints extraídos: {len(fingerprints)}")
print(f"\nMédia por classificação:")
print(fingerprints.groupby("classificacao")[DIMS].mean().round(3))


In [ ]:
# Célula E2 — Radar charts de fingerprints (usa KNOWN_EVENTS)
from math import pi

def radar_chart(ax, values, labels, title, color="steelblue"):
    N = len(labels)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    values = list(values) + [values[0]]
    ax.plot(angles, values, 'o-', linewidth=1.5, color=color)
    ax.fill(angles, values, alpha=0.25, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=8)
    ax.set_title(title, size=10, pad=15)
    ax.set_ylim(0, 1.05)

# Radar por classificação (3 níveis)
classes_info = [("global", "red"), ("nacional", "darkorange"), ("isolado", "steelblue")]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw=dict(polar=True))
for ax, (cls, color) in zip(axes, classes_info):
    sub = fingerprints[fingerprints["classificacao"] == cls]
    if len(sub) > 0:
        vals = sub[DIMS].mean().values
        radar_chart(ax, vals, DIMS, f"{cls.upper()}\n(n={len(sub)})", color=color)
    else:
        ax.set_title(f"{cls.upper()}\n(sem dados)")
plt.suptitle("Fingerprint Médio por Classificação (4 dims × 3 níveis)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "radar_fingerprints.png", dpi=150)
plt.show()

# Radar por evento conhecido (KNOWN_EVENTS)
unique_events = {}
for ev in KNOWN_EVENTS:
    if "start" not in ev or "end" not in ev:
        continue
    if ev["name"] not in unique_events:
        unique_events[ev["name"]] = ev
unique_ev_list = list(unique_events.values())

n_ev = len(unique_ev_list)
if n_ev > 0:
    cols = min(3, n_ev)
    rows = (n_ev + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows),
                             subplot_kw=dict(polar=True))
    axes_flat = np.array(axes).flatten() if n_ev > 1 else [axes]
    for i, ev in enumerate(unique_ev_list):
        mask = (
            (fingerprints["date"] >= pd.Timestamp(ev["start"]))
            & (fingerprints["date"] <= pd.Timestamp(ev["end"]))
        )
        if "portos_foco" in ev and ev["portos_foco"]:
            mask = mask & fingerprints["porto"].isin(ev["portos_foco"])
        sub = fingerprints[mask]
        color_map = {"global": "red", "nacional": "darkorange", "isolado": "steelblue"}
        color = color_map.get(ev["expected"], "gray")
        if len(sub) > 0:
            vals = sub[DIMS].mean().values
            radar_chart(axes_flat[i], vals, DIMS,
                        f"{ev['name'][:25]}\n({ev['expected']})", color)
        else:
            axes_flat[i].set_title(f"{ev['name'][:25]}\n(sem dados)")
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.suptitle("Fingerprints — Eventos Conhecidos (4 dims)", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUT / "radar_eventos.png", dpi=150)
    plt.show()

In [ ]:
# Célula E3 — Classificador endógeno (LOO) com DecisionTree
# Fix 1: 4 dims → Fix 3: multi-class (isolado/nacional/global)
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report

classes_present = fingerprints["classificacao"].nunique()
if classes_present >= 2:
    X = fingerprints[DIMS].values
    y = fingerprints["classificacao"].values
    class_names = sorted(fingerprints["classificacao"].unique())

    loo = LeaveOneOut()
    y_pred = np.empty_like(y)

    for train_idx, test_idx in loo.split(X):
        clf = DecisionTreeClassifier(max_depth=3, random_state=42)
        clf.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = clf.predict(X[test_idx])

    print(f"Classificador LOO — DecisionTree(max_depth=3), {classes_present} classes:")
    print(classification_report(y, y_pred, target_names=class_names))

    # Feature importances (4 dims)
    clf_full = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
    imp = pd.Series(clf_full.feature_importances_, index=DIMS).sort_values(ascending=False)
    print("Feature importances (4 dims):")
    print(imp.round(3))
else:
    print(f"⚠️ Apenas {classes_present} classe(s) — LOO não aplicável.")

## ═══ SEÇÃO F: VISUALIZAÇÕES E ESTUDOS DE CASO ═══


In [ ]:
# Célula G1 — Heatmap de anomalias por porto × ano
# Fix 3: 3 classes
pivot = scores.copy()
pivot["year"] = pivot["date"].dt.year

heat = pivot.groupby(["porto", "year"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(heat, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_title("Anomalias Detectadas por Porto × Ano", fontsize=13)
ax.set_xlabel("Ano")
ax.set_ylabel("Porto")
plt.tight_layout()
plt.savefig(OUT / "heatmap_anomalias.png", dpi=150)
plt.show()

# Heatmap por classificação (3 níveis)
cmaps = {"global": "YlOrRd", "nacional": "Oranges", "isolado": "Blues"}
for cls in ["global", "nacional", "isolado"]:
    sub = pivot[pivot["classificacao"] == cls]
    heat_c = sub.groupby(["porto", "year"]).size().unstack(fill_value=0)
    if heat_c.empty:
        print(f"{cls}: nenhuma anomalia")
        continue
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(heat_c, annot=True, fmt="d", cmap=cmaps.get(cls, "YlOrRd"),
                ax=ax, linewidths=0.5, linecolor="white")
    ax.set_title(f"Anomalias {cls.upper()} por Porto × Ano", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUT / f"heatmap_{cls}.png", dpi=150)
    plt.show()

### F2 — Timeline por porto

Série temporal observada com marcadores de anomalias classificadas. Os marcadores são plotados **no valor observado** ($y_t$) da série na data da anomalia, com cor/forma indicando a classificação.


In [ ]:
# Célula F2 — Timeline por porto (usa KNOWN_EVENTS)
top_portos = scores["porto"].value_counts().head(12).index.tolist()

colors_cls3 = {"global": "red", "nacional": "darkorange", "isolado": "steelblue"}
markers_cls3 = {"global": "v", "nacional": "D", "isolado": "o"}

for porto in top_portos:
    fig, axes = plt.subplots(len(DIMS), 1, figsize=(16, 3 * len(DIMS)), sharex=True)
    if len(DIMS) == 1:
        axes = [axes]

    for ax, dim in zip(axes, DIMS):
        serie_r = resid[(resid["porto"] == porto) & (resid["dim"] == dim)].sort_values("date")
        serie_r = serie_r.dropna(subset=["y_true"])
        ax.plot(serie_r["date"], serie_r["y_true"], color="gray", alpha=0.5, lw=0.8)

        anom = scores[(scores["porto"] == porto) & (scores["dim"] == dim)]
        for cls in ["global", "nacional", "isolado"]:
            sub = anom[anom["classificacao"] == cls]
            if len(sub) > 0:
                merged = sub.merge(
                    resid[["date", "porto", "dim", "y_true"]],
                    on=["date", "porto", "dim"], how="left"
                )
                ax.scatter(merged["date"], merged["y_true"],
                          color=colors_cls3[cls], marker=markers_cls3[cls],
                          s=40, alpha=0.7, label=cls, zorder=3)

        # Sombreamento de eventos conhecidos
        for ev in KNOWN_EVENTS:
            if "start" not in ev or "end" not in ev:
                continue
            portos_foco = ev.get("portos_foco", [])
            if not portos_foco or porto in portos_foco:
                ax.axvspan(pd.Timestamp(ev["start"]), pd.Timestamp(ev["end"]),
                          alpha=0.15, color="gold")

        ax.set_ylabel(dim, fontsize=9)
        if ax == axes[0]:
            ax.legend(fontsize=7, loc="upper right")

    axes[0].set_title(f"Timeline — {porto} (4 dims)", fontsize=12)
    axes[-1].set_xlabel("Data")
    plt.tight_layout()
    plt.savefig(OUT / f"timeline_{porto.replace(' ', '_')}.png", dpi=150)
    plt.show()

### F3 — Estudos de caso (eventos conhecidos)


In [ ]:
# Célula G3 — Estudos de caso (eventos conhecidos - KNOWN_EVENTS)
print("=" * 70)
print("ESTUDOS DE CASO — EVENTOS CONHECIDOS (3 níveis, 4 dims)")
print("=" * 70)

for ev in KNOWN_EVENTS:
    if "start" not in ev or "end" not in ev:
        continue
    portos_foco = ev.get("portos_foco", [])
    portos_str = ", ".join(portos_foco) if portos_foco else "todos"

    print(f"\n{'_' * 60}")
    print(f"  {ev['name']}")
    print(f"   Período: {ev['start']} -> {ev['end']}")
    print(f"   Portos foco: {portos_str}")
    print(f"   Tipo esperado: {ev['expected']}")

    mask = (
        (scores["date"] >= pd.Timestamp(ev["start"]))
        & (scores["date"] <= pd.Timestamp(ev["end"]))
    )
    if portos_foco:
        mask = mask & scores["porto"].isin(portos_foco)
    sub = scores[mask]

    if len(sub) == 0:
        print("   Nenhuma anomalia detectada no período.")
        continue

    print(f"   Anomalias detectadas: {len(sub)}")
    print(f"   Classificação modal: {sub['classificacao'].mode().iloc[0]}")
    print(f"   Score A — min: {sub['score_a'].min()}, max: {sub['score_a'].max()}, "
          f"média: {sub['score_a'].mean():.1f}")
    print(f"   Score B — min: {sub['score_b'].min():.2f}, max: {sub['score_b'].max():.2f}, "
          f"média: {sub['score_b'].mean():.2f}")
    print(f"   Dimensões afetadas: {sub['dim'].unique().tolist()}")

    fp_ev = fingerprints[
        (fingerprints["date"] >= pd.Timestamp(ev["start"]))
        & (fingerprints["date"] <= pd.Timestamp(ev["end"]))
    ]
    if portos_foco:
        fp_ev = fp_ev[fp_ev["porto"].isin(portos_foco)]
    if len(fp_ev) > 0:
        print(f"   Fingerprint médio: {fp_ev[DIMS].mean().round(2).to_dict()}")

### F4 — Ranking de vulnerabilidade


In [ ]:
# Célula G4 — Ranking de vulnerabilidade dos portos
# Fix 3: 3 classes
vuln = scores.groupby("porto").agg(
    total_anomalias=("date", "size"),
    globais=("classificacao", lambda x: (x == "global").sum()),
    nacionais=("classificacao", lambda x: (x == "nacional").sum()),
    isoladas=("classificacao", lambda x: (x == "isolado").sum()),
    score_a_medio=("score_a", "mean"),
    score_b_medio=("score_b", "mean"),
    dims_afetadas=("dim", "nunique"),
    primeiro=("date", "min"),
    ultimo=("date", "max"),
).sort_values("total_anomalias", ascending=False)

vuln["pct_global"] = (100 * vuln["globais"] / vuln["total_anomalias"]).round(1)
vuln["vulnerabilidade"] = (
    vuln["total_anomalias"] * 0.2
    + vuln["globais"] * 0.4
    + vuln["nacionais"] * 0.2
    + vuln["dims_afetadas"] * 0.2
).round(2)
vuln = vuln.sort_values("vulnerabilidade", ascending=False)

print("RANKING DE VULNERABILIDADE DOS PORTOS")
print("=" * 70)
print(vuln[["total_anomalias", "globais", "nacionais", "isoladas",
            "pct_global", "dims_afetadas", "vulnerabilidade"]].to_string())

# Gráfico de barras
fig, ax = plt.subplots(figsize=(12, 8))
vuln_plot = vuln.head(TOP_N_PORTOS)
bars = ax.barh(vuln_plot.index, vuln_plot["vulnerabilidade"], color="coral")
ax.set_xlabel("Índice de Vulnerabilidade")
ax.set_title(f"Ranking de Vulnerabilidade dos Portos ({TOP_N_PORTOS} portos, 3 níveis)", fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT / "ranking_vulnerabilidade.png", dpi=150)
plt.show()

## ═══ SEÇÃO G: EXPORTAÇÃO DE RESULTADOS ═══


In [ ]:
# Célula G5 — Salvar resultados finais
scores.to_parquet(OUT / "anomalias_classificadas.parquet", index=False)
scores.to_csv(OUT / "anomalias_classificadas.csv", index=False)
print(f"✅ anomalias_classificadas: {len(scores)} linhas")

fingerprints.to_parquet(OUT / "fingerprints.parquet", index=False)
fingerprints.to_csv(OUT / "fingerprints.csv", index=False)
print(f"✅ fingerprints: {len(fingerprints)} linhas")

# Resíduos completos (para NB3 exploração)
resid.to_parquet(OUT / "residuos.parquet", index=False)
print(f"✅ residuos: {len(resid)} linhas")

vuln.to_csv(OUT / "ranking_vulnerabilidade.csv")
print(f"✅ ranking_vulnerabilidade: {len(vuln)} portos")

# Threshold adaptativo por porto
astat.to_csv(OUT / "threshold_adaptativo.csv", index=False)
print(f"✅ threshold_adaptativo: {len(astat)} linhas")

# Near-misses (quase-anomalias)
if len(near_misses) > 0:
    near_misses.to_parquet(OUT / "near_misses.parquet", index=False)
    near_misses.to_csv(OUT / "near_misses.csv", index=False)
    print(f"✅ near_misses: {len(near_misses)} linhas")
else:
    print("⚠️ Nenhum near-miss para exportar")

grid_df.to_csv(OUT / "grid_search_dual_score.csv", index=False)
print(f"✅ grid_search_dual_score: {len(grid_df)} combinações")

# Resumo geral
print(f"\n{'=' * 60}")
print(f"RESUMO FINAL (v4.2 — ta=3 forçado, tb=0.8)")
print(f"{'=' * 60}")
print(f"Dimensões modeladas: {len(DIMS)} ({DIMS})")
print(f"Total de anomalias: {len(scores)}")
n_gl = (scores['classificacao'] == 'global').sum()
n_na = (scores['classificacao'] == 'nacional').sum()
n_is = (scores['classificacao'] == 'isolado').sum()
print(f"  Global: {n_gl}  |  Nacional: {n_na}  |  Isolado: {n_is}")
print(f"Portos analisados: {scores['porto'].nunique()}")
print(f"Período: {scores['date'].min()} → {scores['date'].max()}")
print(f"Limiar forçado: ta={BEST_TA}, tb={BEST_TB}")
print(f"Grid search indicou: ta={int(best['ta'])}, tb={float(best['tb'])}, recall={best['recall']}")
print(f"Precision: {precision:.2f} ({hits_df['acerto'].sum()}/{len(hits_df)} classificações corretas)")
print()
print(f"Taxa de alertas: {taxa_total:.1%} das observações sinalizadas")
print(f"  Global:   {n_gl/total_obs:.2%}")
print(f"  Nacional: {n_na/total_obs:.2%}")
print(f"  Isolado:  {n_is/total_obs:.2%}")
print(f"\nArquivos salvos em: {OUT}")
print(f"Figuras: {[f.name for f in OUT.glob('*.png')]}")